# **Data Integration**

In [11]:
# import necessary libraries
import pandas as pd

## **Results and Sprints**

In [12]:
df_results = pd.read_csv("../data_cleaning/results_api_clean.csv")
df_results["isSprint"] = 0

df_sprint = pd.read_csv("../data_cleaning/sprints_api_clean.csv")
df_sprint["isSprint"] = 1

# Unisce gare normali e Sprint
df_all = pd.concat([df_results, df_sprint], ignore_index=True)

# Salva tutto in un CSV unico
df_all.to_csv("results.csv", index=False)
#print(f"Totale risultati 2025 (gare + sprint): {len(df_all_2025)}")

df_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 599 entries, 0 to 598
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   round          599 non-null    int64
 1   driverId       599 non-null    str  
 2   constructorId  599 non-null    str  
 3   grid           599 non-null    int64
 4   position       599 non-null    int64
 5   points         599 non-null    int64
 6   status         599 non-null    str  
 7   laps           599 non-null    int64
 8   time           533 non-null    str  
 9   fastestLap     573 non-null    str  
 10  isSprint       599 non-null    int64
dtypes: int64(6), str(5)
memory usage: 88.1 KB


## **Drivers**

In [13]:
df_wiki = pd.read_csv('../data_cleaning/drivers_wiki_clean.csv')
df_api = pd.read_csv('../data_cleaning/drivers_api_clean.csv')

#find surname in driverName and merge
merged_rows = []
for _, wiki_row in df_wiki.iterrows():
    for _, api_row in df_api.iterrows():
        if api_row['surname'].lower() in wiki_row['driverName'].lower():
            merged_row = api_row.to_dict()
            merged_row.update(wiki_row.to_dict())
            merged_row.pop('driverName', None)
            merged_rows.append(merged_row)
            break

df_drivers = pd.DataFrame(merged_rows)
df_drivers.to_csv('drivers.csv', index=False)

## **Constructors**

In [14]:
df_wiki = pd.read_csv("../data_cleaning/constructor_wiki_clean.csv")
df_api = pd.read_csv("../data_acquisition/constructors_api.csv")

# alphabetically order both dataframes by constructor name
df_wiki = df_wiki.sort_values("constructor").reset_index(drop=True)
df_api = df_api.sort_values("constructorName").reset_index(drop=True)

# add constructorId from df_api to df_wiki based on row-by-row matching
df_wiki.insert(0, "constructorId", df_api["constructorId"])

df_wiki.to_csv("constructors.csv", index=False)

## **Circuits**

In [15]:
df_races = pd.read_csv('../data_cleaning/races.csv') 
df_circuits = pd.read_csv('../data_cleaning/circuits_wiki_clean.csv')

circuit_ids = [] # array of ids ordered as df_circuits
for _, circuit_row in df_circuits.iterrows():
    grandPrix = circuit_row['grandPrix']
    found = None

    # look for exact match first
    for _, race_row in df_races.iterrows():
        raceName = race_row['raceName']

        if raceName == grandPrix:
            found = race_row['circuitId']
            break

    # if not found, look for partial match
    if found is None:
        for _, race_row in df_races.iterrows():
            raceName = race_row['raceName']
            if raceName in grandPrix:
                found = race_row['circuitId']
                break

    circuit_ids.append(found)

# add id column as the first column
df_circuits.insert(0, 'circuitId', circuit_ids)

# drop grandPrix column
df_circuits = df_circuits.drop(columns=['grandPrix'])

df_circuits.to_csv('circuits.csv', index=False)